In [2]:
import pandas as pd
import requests

# Load GTFS shapes.txt
df = pd.read_csv("gtfsAlex/shapes.txt")
prices = []

# Group by shape_id
for shape_id, group in df.groupby("shape_id"):
    # ensure correct order
    group = group.sort_values("shape_pt_sequence")
    coords = ";".join(f"{lon},{lat}" for lat, lon in zip(group["shape_pt_lat"], group["shape_pt_lon"]))
    url = f"http://router.project-osrm.org/route/v1/driving/{coords}?overview=false"
    print(url)
    try:
        resp = requests.get(url, timeout=30)
        resp.raise_for_status()
        res = resp.json()
    except requests.exceptions.RequestException as e:
        print(f"Request error for {shape_id}: {e}")
        continue
    except ValueError as e:
        print(f"JSON decode error for {shape_id}: {e}")
        continue

    if res.get("code") != "Ok" or "routes" not in res or not res["routes"]:
        print(f"OSRM error for {shape_id}: {res.get('message', res.get('code'))}")
        continue

    try:
        distance_m = res["routes"][0]["distance"]
    except Exception as e:
        print(f"Parsing error for {shape_id}: {e}")
        continue

    km = distance_m / 1000
    print(shape_id, f"{km:.4f} km")
    prices.append((shape_id, km))

# save mapping to CSV
out_df = pd.DataFrame(prices, columns=["shape_id", "distance_km"])
out_df.to_csv("shape_distances.csv", index=False)
print(f"Saved {len(out_df)} shape distances to shape_distances.csv")



http://router.project-osrm.org/route/v1/driving/29.992657,31.219638;29.990762,31.220545;29.989934,31.221025;29.988078,31.222316;29.987604,31.222598;29.987217,31.222803;29.986609,31.223073;29.983906,31.223975;29.98374,31.223977;29.983628,31.223949;29.983261,31.22405;29.982266,31.22438;29.98203,31.22452;29.981489,31.224759;29.980825,31.225009;29.980539,31.225104;29.979542,31.225356;29.979215,31.225387;29.979019,31.225328;29.977086,31.224236;29.976737,31.224088;29.976373,31.223967;29.975999,31.223869;29.975246,31.22373;29.975001,31.224967;29.974965,31.225329;29.974946,31.225373;29.97487,31.225451;29.974662,31.227287;29.974623,31.22753;29.974536,31.227776;29.974393,31.228026;29.973988,31.228638;29.973205,31.229482;29.972046,31.230842?overview=false
-Q2gVX9GgVSEtVr-yv6Nf_Shape 6.2782 km
http://router.project-osrm.org/route/v1/driving/29.915571,31.175507;29.917287,31.176185;29.919028,31.17684;29.919254,31.176912;29.91988,31.177069;29.920247,31.177217;29.920748,31.177494;29.921161,31.177686;2

In [5]:
import requests
import pandas as pd

API_KEY = "5b3ce3597851110001cf6248d66aad82c6ed4679b540bf0fd54b4f5d"

df = pd.read_csv("gtfsAlex/shapes.txt")
df = df[df['shape_id'].isin(['wzQQv79eeOAEsj4p6t-wx_Shape'])]
test = []
for shape_id, group in df.groupby("shape_id"):
    coords = [[lon, lat] for lat, lon in zip(group["shape_pt_lat"], group["shape_pt_lon"])]
    body = {"coordinates": coords}
    res = requests.post(
        f"https://api.openrouteservice.org/v2/directions/driving-car",
        headers={"Authorization": API_KEY, "Content-Type": "application/json"},
        json=body
    ).json()
    print (res)


{'error': {'code': 2004, 'message': 'Request parameters exceed the server configuration limits. The specified number of waypoints must not be greater than 70.'}, 'info': {'engine': {'build_date': '2025-06-06T15:39:25Z', 'graph_version': '2', 'graph_date': '2025-10-12T11:31:24Z', 'osm_date': '2025-10-06T00:00:00Z', 'version': '9.3.0'}, 'timestamp': 1760884806770}}


In [13]:
import pandas as pd
import requests
from tqdm import tqdm
# === Load GTFS files ===
stops = pd.read_csv("gtfsAlex/stops.txt")
stop_times = pd.read_csv("gtfsAlex/stop_times.txt")
trips = pd.read_csv("gtfsAlex/trips.txt")

# Merge to get coordinates
stop_times = stop_times.merge(stops[["stop_id", "stop_lat", "stop_lon"]], on="stop_id")

# Sort by trip_id and stop_sequence
stop_times = stop_times.sort_values(["trip_id", "stop_sequence"])

results = []

for trip_id, group in tqdm(stop_times.groupby("trip_id")):
    group = group.reset_index(drop=True)
    for i in range(len(group) - 1):
        s1 = group.iloc[i]
        s2 = group.iloc[i + 1]

        # Coordinates for OSRM
        coords = f"{s1['stop_lon']},{s1['stop_lat']};{s2['stop_lon']},{s2['stop_lat']}"
        url = f"http://router.project-osrm.org/route/v1/driving/{coords}?overview=false"

        try:
            r = requests.get(url, timeout=10).json()
            if "routes" in r:
                dist_m = r["routes"][0]["distance"]
                results.append({
                    "trip_id": trip_id,
                    "from_stop_id": s1["stop_id"],
                    "to_stop_id": s2["stop_id"],
                    "distance_km": dist_m / 1000
                })
        except Exception as e:
            print(f"Error for {trip_id}: {e}")

df_out = pd.DataFrame(results)
df_out.to_csv("trip_stop_distances.csv", index=False)
print("✅ Saved trip_stop_distances.csv")


  0%|          | 0/192 [00:00<?, ?it/s]

 16%|█▌        | 30/192 [04:53<15:34,  5.77s/it]

Error for 8wpT6YlGJmfsskCDg8AJT-07:00:00: HTTPConnectionPool(host='router.project-osrm.org', port=80): Read timed out. (read timeout=10)


 27%|██▋       | 51/192 [09:02<31:54, 13.58s/it]

Error for FUo5FExiKwUTpyTUJYA7R-07:00:00: HTTPConnectionPool(host='router.project-osrm.org', port=80): Read timed out. (read timeout=10)
Error for FUo5FExiKwUTpyTUJYA7R-07:00:00: HTTPConnectionPool(host='router.project-osrm.org', port=80): Max retries exceeded with url: /route/v1/driving/29.97830430085358,31.247678247244345;29.96426186955863,31.23965035567426?overview=false (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000029F73DE5810>: Failed to establish a new connection: [WinError 10051] A socket operation was attempted to an unreachable network'))


...

Error for _npHlyCCY7o0R20RyqvT8-07:00:00: HTTPConnectionPool(host='router.project-osrm.org', port=80): Read timed out. (read timeout=10)


 83%|████████▎ | 160/192 [37:08<04:18,  8.08s/it]

Error for t3fihouAmNU32d_e7msfe-07:00:00: HTTPConnectionPool(host='router.project-osrm.org', port=80): Max retries exceeded with url: /route/v1/driving/30.03500306571987,31.284028419006447;30.027416,31.283503?overview=false (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x0000029F73E5A3D0>: Failed to establish a new connection: [WinError 10051] A socket operation was attempted to an unreachable network'))


100%|██████████| 192/192 [1:08:00<00:00, 21.25s/it]   


✅ Saved trip_stop_distances.csv


In [14]:
# aggregate distances per trip
trip_distances = df_out.groupby("trip_id")["distance_km"].sum().reset_index()
trip_distances.to_csv("trip_distances.csv", index=False)